<div style="border-radius: 10px; padding: 32px 0px; border: 1px solid rgba(128,128,128,0.2);">
  <div style="display: flex; justify-content: space-between; align-items: flex-start; flex-wrap: wrap; gap: 16px; padding: 0px 32px;">
  <div>
    <div style="font-size: 0.75rem; letter-spacing: 3px; text-transform: uppercase; font-weight: 600; margin-bottom: 10px; opacity: 0.6;">
      Máster Universitario en Big Data y Computación en la Nube.
    </div>
    <div style="font-size: 1.5rem; font-weight: 700; margin-bottom: 4px;">Trabajo de Fin de Máster</div>
    <div style="font-size: 1rem; font-weight: 400; opacity: 0.75;">Clasificador taxonómico de boletines oficiales españoles</div>
  </div>
  <div style="margin-top: 16px; display: flex; align-items: center; gap: 12px;">
    <div style="font-size: 1rem; font-weight: 600;">Hugo de Lamo</div>
  </div>
  </div>
</div>

# 05 · Clasificador taxonómico de boletines oficiales

Este notebook implementa un clasificador multietiqueta de publicaciones de boletines oficiales españoles usando **Pydantic AI**. El problema es una aguja en un pajar: de ~65 000 publicaciones del Q1 2025, solo ~3,7 % son relevantes para el dominio ambiental-energético.

El clasificador responde cuatro preguntas por publicación:
1. **¿Es relevante?** - ¿Pertenece al universo de autorizaciones ambiental-energéticas?
2. **¿Qué procedimientos contiene?** - Lista multilabel: DIA, AAP, AAC, AAU, IIA, AAI, IAE, DUP.
3. **¿Qué tipo de acto es?** - Forma jurídica del documento (N1): resolución, anuncio, decreto…
4. **¿Qué tecnología menciona?** - Lista multilabel: fotovoltaica, eólica, hidrógeno…

---

## Estructura del notebook

### Parte I - Fundamentos
|   | Sección | Contenido |
|---|---------|----------|
| **0** | **Setup** | Entorno, dependencias, modelo local |
| **1** | **Schema de output** | `ClassifierOutput`, enums e invariantes |
| **2** | **Pre-procesamiento** | N0 lookup + N1 clasificador por reglas |
| **3** | **Ground truth** | Muestreo estratificado + anotación manual |
| **4** | **Agente base** | System prompt, construcción y casos cualitativos |

### Parte II - Ciclo de experimentación
|   | Sección | Contenido |
|---|---------|----------|
| **5** | **Experimento 1 - Baseline** | Zero-shot, sin contexto N1 |
| **6** | **Análisis de errores** | Qué falla y por qué |
| **7** | **Prompt v2** | Mejora basada en errores (DEC-013 a DEC-019) |
| **8** | **Experimento 2 - Prompt v2** | ¿Mejora respecto a baseline? |
| **9** | **Experimento 3 - Ablación +N1** | ¿Aporta el contexto de forma? |
| **10** | **Experimento 4 - Few-shot** | ¿Ayudan los ejemplos reales? |
| **11** | **Comparativa de modelos** | Qwen 3.5 9B vs Gemma 4 4B |

### Parte III - Análisis final
|   | Sección | Contenido |
|---|---------|----------|
| **12** | **Tabla resumen** | Comparativa de todos los experimentos |
| **13** | **Calibración de confianza** | ¿El modelo sabe cuándo no sabe? |
| **14** | **Conclusiones y trabajo futuro** | Hallazgos, limitaciones, v2 |

---

##  0. Setup

Cargamos las variables de entorno e importamos las librerías. El modelo vive en LM Studio - el único punto de cambio para conectar otro proveedor es `LM_STUDIO_MODEL`.

In [ ]:
# Librerías estándar
import os
import html
import re
import json
import asyncio
from pathlib import Path

# Librerías de datos
import pandas as pd

# Variables de entorno (.env buscado desde el directorio del proyecto)
from dotenv import find_dotenv, load_dotenv

# Pydantic AI - framework de agentes con output estructurado
from pydantic_ai import Agent

load_dotenv(find_dotenv())


In [ ]:
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider

# Modelo local via LM Studio 
# Cambiar LM_STUDIO_MODEL según el modelo cargado en LM Studio
LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

print(f"Modelo: {LM_STUDIO_MODEL} via LM Studio (localhost:1234)")


In [ ]:
import httpx

# Verificar que LM Studio responde antes de ejecutar experimentos
try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    print(f"LM Studio operativo.")
    print(f"Modelos disponibles: {modelos}")
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado"
    print(f"✓ {LM_STUDIO_MODEL} listo")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - ¿está el servidor arrancado?")


---

##  1. Schema de output

El schema define el **contrato entre el LLM y el sistema**: qué campos devuelve el modelo, de qué tipo y bajo qué restricciones. Pydantic valida cada respuesta antes de que llegue al resto del código, forzando un retry automático si algo no cumple el schema.

`ClassifierOutput` tiene 6 campos:

| Campo | Tipo | Rol |
|-------|------|-----|
| `is_relevant` | `bool` | ¿Pertenece al dominio ambiental-energético? |
| `act_type` | `ActType` | Forma jurídica del acto (N1) - resolución, anuncio, decreto… |
| `procedures` | `list[ProcedureType]` | Procedimientos identificados (N2) - multilabel |
| `technologies` | `list[TechnologyType]` | Tecnologías mencionadas (N3) - multilabel, puede ser vacía |
| `confidence` | `float` | Confianza global entre 0.0 y 1.0 |
| `reasoning` | `str` | Justificación breve citando el texto que dispara cada etiqueta |

Un `@model_validator` impone los invariantes de negocio: `is_relevant=True` exige `procedures != []`; `is_relevant=False` exige ambas listas vacías.

In [ ]:
# Importar schema de output: enums y modelo Pydantic que define qué devuelve el agente
from clasificador.schema import ActType, ProcedureType, TechnologyType, ClassifierOutput


In [13]:
# Verificación de invariantes
ejemplo_valido = ClassifierOutput(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    procedures=[ProcedureType.DIA],
    technologies=[TechnologyType.FOTOVOLTAICA],
    confidence=0.98,
    reasoning="'se formula la declaración de impacto ambiental' → DIA. 'Planta Solar Fotovoltaica' → fotovoltaica.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutput(
        is_relevant=False, act_type=ActType.RESOLUCION,
        procedures=[ProcedureType.AAP], technologies=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError → {e.errors()[0]['msg']}")

Ejemplo válido:
{
  "is_relevant": true,
  "act_type": "resolución",
  "procedures": [
    "DIA"
  ],
  "technologies": [
    "fotovoltaica"
  ],
  "confidence": 0.98,
  "reasoning": "'se formula la declaración de impacto ambiental' → DIA. 'Planta Solar Fotovoltaica' → fotovoltaica."
}

Violación de invariante:
  ValidationError → Value error, is_relevant=False con procedures != []


---

##  2. Pre-procesamiento

Antes de llamar al LLM, cada registro pasa por dos pasos deterministas:

- **N0 - Ámbito**: lookup directo del campo `bulletin` → `estatal / autonómico / local`. Sin LLM.
- **N1 - Tipo de acto**: clasificador de primer token con pre-procesamiento de formatos especiales (BOCM, BOCA, BOE topónimos).

**Cobertura real medida**: 88.6% del corpus. El 11.4% restante cae en `OTROS` - principalmente topónimos BOE irrecuperables sin PDF.

In [ ]:
# Cargar el corpus completo de boletines oficiales Q1 2025
PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"

df = pd.read_parquet(PATH_PARQUET)

# Decodificar entidades HTML que vienen del scraping (ej. &amp; → &)
df["description"] = df["description"].apply(html.unescape)

print(f"Corpus: {len(df):,} registros · {df['bulletin'].nunique()} boletines")


In [ ]:
# N0: Ámbito geográfico por boletín 
# Mapeo de identificadores de boletín a su ámbito (estatal / autonómico / local).
# Los boletines no listados aquí se clasifican como 'autonómico' por defecto.
_GAZETTE_TO_AMBITO = {
    "boe": "estatal",
    "madridambiental": "local",
    # resto → autonómico por defecto
}

def get_ambito(bulletin: str) -> str:
    """Devuelve el ámbito geográfico del boletín."""
    return _GAZETTE_TO_AMBITO.get(bulletin.lower(), "autonómico")


In [ ]:
# N1: Pre-procesamiento y mapa de patrones 
# preprocess_description limpia el texto antes de aplicar el clasificador N1:
# - elimina prefijos de sección (líneas con '–' o '-')
# - detecta topónimos puros y subtítulos de subastas de la AEAT
def preprocess_description(desc: str, bulletin: str) -> str:
    desc = desc.strip()
    # Algunos boletines añaden una línea de sección antes de la descripción real
    if "\n–" in desc:
        desc = desc.split("\n–", 1)[1].strip()
    elif "\n-" in desc:
        desc = desc.split("\n-", 1)[1].strip()
    # BOCA y BOIB usan '.-' como separador de sección
    if bulletin.lower() in ("boca", "boib") and ".-" in desc:
        desc = desc.split(".-", 1)[1].strip()
    # Detectar topónimo puro (todo mayúsculas, ≤4 palabras)
    if re.match(r"^[A-ZÁÉÍÓÚÜÑ/\s]+$", desc) and len(desc.split()) <= 4:
        return "__TOPONIMO__"
    # Detectar anuncios de subasta de la AEAT
    if desc.upper().startswith(("U.R.", "E.R.", "SUMA GESTIÓN", "ORGANISMO AUTÓNOMO DE HACIENDA")):
        return "__SUBASTA_AEAT__"
    return desc


# Tabla de patrones N1: (prefijo_en_minúsculas, ActType)
# El orden importa - se aplica el primer patrón que coincide.
_N1_MAP = [
    (r"corrección de errat",     ActType.CORRECCION_ERRORES),
    (r"corrección de error",     ActType.CORRECCION_ERRORES),
    (r"rectificación",           ActType.CORRECCION_ERRORES),
    (r"real decreto",            ActType.REAL_DECRETO),
    (r"orden foral",             ActType.ORDEN),
    (r"información pública",     ActType.INFORMACION_PUBLICA),
    (r"exposición pública",      ActType.INFORMACION_PUBLICA),
    (r"trámite de información",  ActType.INFORMACION_PUBLICA),
    (r"resolución",              ActType.RESOLUCION),
    (r"anuncio",                 ActType.ANUNCIO),
    (r"orden",                   ActType.ORDEN),
    (r"decreto foral",           ActType.DECRETO),
    (r"decreto",                 ActType.DECRETO),
    (r"acuerdo",                 ActType.ACUERDO),
    (r"aprobación",              ActType.APROBACION),
    (r"extracto",                ActType.EXTRACTO),
    (r"convenio",                ActType.CONVENIO),
    (r"adenda",                  ActType.CONVENIO),
    (r"solicitud",               ActType.SOLICITUD),
    (r"modificación",            ActType.MODIFICACION),
    (r"edicto",                  ActType.EDICTO),
    (r"notificación",            ActType.NOTIFICACION),
    (r"notificaciones",          ActType.NOTIFICACION),
    (r"recaudación ejecutiva",   ActType.NOTIFICACION),
    (r"propuesta de resolución", ActType.RESOLUCION),
    (r"bases",                   ActType.CONVOCATORIA),
    (r"convocatoria",            ActType.CONVOCATORIA),
    (r"nombramiento",            ActType.RESOLUCION),
    (r"delegación",              ActType.RESOLUCION),
    (r"emplazamiento",           ActType.NOTIFICACION),
    (r"citación",                ActType.NOTIFICACION),
    (r"diligencia",              ActType.NOTIFICACION),
    (r"cédula",                  ActType.NOTIFICACION),
    (r"requerimiento",           ActType.NOTIFICACION),
    (r"sala primera",            ActType.OTROS),
    (r"sala segunda",            ActType.OTROS),
    (r"__toponimo__",            ActType.OTROS),
    (r"__subasta_aeat__",        ActType.OTROS),
    (r"concesión",               ActType.RESOLUCION),
    (r"trámite de audiencia",    ActType.INFORMACION_PUBLICA),
    (r"trámite de",              ActType.INFORMACION_PUBLICA),
    (r"iniciación",              ActType.RESOLUCION),
    (r"inicio",                  ActType.RESOLUCION),
    (r"apertura",                ActType.RESOLUCION),
    (r"informe",                 ActType.RESOLUCION),
    (r"expediente",              ActType.RESOLUCION),
    (r"ley",                     ActType.OTROS),        # disposiciones normativas
    (r"recurso",                 ActType.RESOLUCION),   # recursos administrativos
    (r"notaría",                 ActType.RESOLUCION),   # actas notariales BON
    (r"publicación",             ActType.ANUNCIO),      # anuncios de publicación
    (r"plan",                    ActType.APROBACION),   # planes aprobados
    (r"departamento",            ActType.RESOLUCION),   # BOIB residual
]

# TODO: revisar casos residuales de "otors" y ver si se pueden reclasificar con patrones adicionales. Ejemplos:

In [ ]:
def inferir_act_type(description: str, bulletin: str) -> ActType:
    """Clasifica el tipo de acto (N1) por coincidencia de prefijo en _N1_MAP."""
    # Limpiar y normalizar el texto antes de comparar
    desc_clean = preprocess_description(description, bulletin)
    text = desc_clean.lower().strip()
    # Recorrer el mapa en orden; gana la primera coincidencia
    for pattern, act_type in _N1_MAP:
        if text.startswith(pattern):
            return act_type
    return ActType.OTROS


In [ ]:
# Aplicar el clasificador N1 a todo el corpus y auditar la cobertura
df["act_type_n1"] = df.apply(
    lambda row: inferir_act_type(row["description"], row["bulletin"]), axis=1
)

dist = df["act_type_n1"].value_counts()
total = len(df)
print("Distribución N1 inferida:\n")
for val, count in dist.items():
    print(f"  {val:<25} {count:>6,}  ({count/total*100:.1f}%)")

# 'OTROS' agrupa todo lo que no encaja en ningún patrón - mide el hueco de cobertura
otros = (df["act_type_n1"] == ActType.OTROS).sum()
print(f"\nCobertura N1: {(1 - otros/total)*100:.1f}%  ({otros:,} en OTROS)")


### Límites del clasificador N1 y trabajo futuro

El clasificador de primer token cubre **89.6%** del corpus con reglas deterministas.
El 10.4% restante cae en `OTROS` por tres motivos con soluciones conocidas:

| Grupo | Volumen aprox. | Motivo | Solución futura |
|---|---|---|---|
| Topónimos BOE / subastas AEAT | ~5.200 | Sin contenido textual real - irrecuperable sin PDF | Clase propia `NO_INFERIBLE` en v2 |
| BON fiscal | ~160 | `tipos`, `calendario` - formatos tributarios navarros | Reglas específicas BON |
| RRHH sin tipo explícito | ~350 | `relación`, `lista`, `oferta`, `bajas` - tipo de acto no en la descripción | LLM zero-shot viable en v2 (DEC-011) |
| BOIB residual | ~300 | Variantes de prefijo catalán no contempladas | Ampliar split `.-` para BOIB |
| Long tail distribuido | ~740 | Tokens poco frecuentes sin patrón claro | Cobertura ~98.5% teórica con PDF |

**Techo práctico estimado sin PDF**: ~98.5% añadiendo más tokens al mapa.
**Techo real con PDF**: ~100% - el tipo de acto aparece siempre en el cuerpo del documento.

---

##  3. Ground truth

El ground truth se construye en tres capas:

| Capa | Qué etiqueta | Cómo | 
|------|-------------|------|
| **1 - N1 determinista** | `act_type` | Reglas de primer token ( 2) | 
| **2 - Muestreo estratificado** | Selección de 100 registros | Keywords por procedimiento N2 | 
| **3 - Anotación manual** | `is_relevant_gt`, `procedures_gt`, `technologies_gt` | Revisión humana |

El archivo `ground_truth_100_anotado.csv` contiene los 100 registros con etiquetas manuales validadas.

In [ ]:
import random
random.seed(42)

# Paso 1: muestreo estratificado 500 registros 
keywords = {
    "DIA":     ["declaración de impacto ambiental"],
    "AAP":     ["autorización administrativa previa"],
    "AAC":     ["autorización de construcción", "autorización administrativa de construcción"],
    "AAP_AAC": ["previa y de construcción"],
    "AAU":     ["autorización ambiental unificada"],
    "IIA":     ["informe de impacto ambiental"],
    "AAI":     ["autorización ambiental integrada"],
    "IAE":     ["ambiental estratégic"],
    "DUP":     ["utilidad pública"],
}
cuotas = {"DIA": 60, "AAP": 60, "AAC": 50, "AAP_AAC": 40,
          "AAU": 40, "IIA": 40, "AAI": 30, "IAE": 30, "DUP": 30}
N_NEGATIVOS = 120

sampled_ids = set()
frames = []
for grupo, kws in keywords.items():
    mask = df["description"].str.lower().str.contains("|".join(kws), na=False)
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas[grupo], len(pool))
    sample = pool.sample(n, random_state=42).copy()
    sample["grupo_muestreo"] = grupo
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {grupo:<10} pool={len(pool):>5,}  sampled={n}")

all_kws = [kw for kws in keywords.values() for kw in kws]
mask_neg = ~df["description"].str.lower().str.contains("|".join(all_kws), na=False)
mask_neg = mask_neg & ~df.index.isin(sampled_ids)
negativos = df[mask_neg].sample(N_NEGATIVOS, random_state=42).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_gt_full = pd.concat(frames, ignore_index=True)
df_gt_full["id"] = range(len(df_gt_full))
print(f"\nPaso 1 - {len(df_gt_full)} registros muestreados")
print(df_gt_full["grupo_muestreo"].value_counts().to_string())


In [ ]:
#  Paso 2: submuestra proporcional de 100 registros 
total = len(df_gt_full)
df_gt_100 = pd.concat([
    g.sample(min(len(g), max(1, round(len(g) * 100 / total))), random_state=42)
    for _, g in df_gt_full.groupby("grupo_muestreo")
]).reset_index(drop=True)

print(f"\nPaso 2 - {len(df_gt_100)} registros en la submuestra:")
print(df_gt_100["grupo_muestreo"].value_counts().to_string())


### Dataset anotado

Los 100 registros del ground truth han sido anotados manualmente. Durante el proceso se identificaron casos especiales documentados en `decisiones_implementacion.md`:

- **Denegaciones**: heredan el tipo de procedimiento del acto denegado
- **Modificaciones**: heredan los procedimientos del acto modificado
- **Falsos positivos**: RRHH con vocabulario ambiental, concesiones de dominio público no energéticas

In [ ]:
# Cargar el CSV anotado manualmente (100 registros con etiquetas ground truth)
PATH_GT_ANOTADO = "../data/ground_truth/ground_truth_100_anotado.csv"
df_anotado = pd.read_csv(PATH_GT_ANOTADO)

print(f"Ground truth anotado: {len(df_anotado)} registros")
print(f"Relevantes:     {df_anotado['is_relevant_gt'].astype(bool).sum()}")
print(f"No relevantes:  {(~df_anotado['is_relevant_gt'].astype(bool)).sum()}")
print(f"\nDistribución N2:")
print(df_anotado["procedures_gt"].value_counts().to_string())


---

##  4. Agente base

El agente Pydantic AI recibe una descripción de boletín y devuelve un `ClassifierOutput` validado. Se compara en dos configuraciones:

- **Baseline**: input = `description` + `bulletin`. El LLM infiere `act_type` desde el texto.
- **+N1**: input = `description` + `bulletin` + `act_type` pre-computado. El LLM lo usa como contexto.

In [ ]:
# System prompt Baseline (Experimento 1)
# Define el rol del agente, el dominio, los procedimientos N2 y las instrucciones de output.
SYSTEM_PROMPT = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles.
Tu tarea es analizar la descripción de una publicación y asignarle etiquetas según la taxonomía definida.

## Dominio
Las publicaciones relevantes pertenecen al universo de autorizaciones ambiental-energéticas (~3.7% del corpus).
El resto (RRHH, contratos, subvenciones, urbanismo...) son is_relevant=False.

## Procedimientos N2
| Etiqueta | Descripción |
|----------|-------------|
| DIA | Declaración de Impacto Ambiental - resolución que formula o aprueba el impacto ambiental |
| AAP | Autorización Administrativa Previa - valida el anteproyecto |
| AAC | Autorización Administrativa de Construcción - permiso definitivo de obras |
| AAU | Autorización Ambiental Unificada - equivalente regional a DIA en BOJA/DOE/BON |
| IIA | Informe de Impacto Ambiental - evaluación simplificada, distinta de DIA |
| AAI | Autorización Ambiental Integrada - permiso IPPC/IED, distinta de DIA |
| IAE | Informe/Declaración Ambiental Estratégico - aplica a planes y programas |
| DUP | Declaración de Utilidad Pública - reconoce interés general, habilita expropiación |

## Tecnologías N3
fotovoltaica · eólica · almacenamiento · hibridación · hidroeléctrica ·
biogás_biometano · biomasa · hidrógeno · línea_eléctrica · gas_natural · petróleo

## Reglas críticas
1. is_relevant=True SOLO si identificas al menos un procedimiento N2
2. AAU ≠ DIA - son procedimientos distintos aunque equivalentes funcionalmente
3. "autorización administrativa previa y de construcción" → [AAP, AAC] (no solo AAP)
4. "aprobación del proyecto de ejecución" junto a AAP → añadir AAC
5. IIA ≠ DIA - el informe de impacto ambiental es evaluación simplificada
6. AAI ≠ DIA - solo es DIA si menciona explícitamente "declaración de impacto ambiental"
7. IAE aplica a planes y programas, no a proyectos individuales
8. DUP puede acompañar a AAP/AAC pero no es AAP ni AAC por sí sola
9. Las denegaciones y desistimientos heredan el tipo del procedimiento denegado
10. Las modificaciones heredan los procedimientos del acto modificado
11. reasoning debe citar el fragmento exacto del texto que dispara cada etiqueta
""".strip()

In [ ]:
# Construir el agente base: model + output estructurado + system prompt
# output_type=ClassifierOutput fuerza al LLM a devolver JSON válido según el schema
agent = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT,
)


`clasificar()` - función de inferencia unitaria

Recibe la descripción de un registro y devuelve un `ClassifierOutput` validado.

**Flujo interno:**

```
bulletin + description
    │
    ├─► get_ambito(bulletin)        → "estatal" / "autonómico" / "local"
    │
    ├─► [si use_n1_context=True]
    │       inferir_act_type()      → añade "Tipo de acto: resolución" al mensaje
    │
    ├─► agent.run(user_msg)         → llamada al LLM (async, espera respuesta)
    │
    └─► result.output              → ClassifierOutput ya validado por Pydantic
```

**Parámetros:**
- `description` - texto del boletín (ya con `html.unescape` aplicado)
- `bulletin` - código del boletín en minúsculas (`"boja"`, `"boe"`...)
- `use_n1_context` - si `True`, incluye el `act_type` pre-computado como contexto extra

**Configuraciones de experimento:**

| `use_n1_context` | Experimento |
|-----------------|-------------|
| `False` | Exp 1 - Baseline / Exp 2 - Prompt v2 |
| `True` | Exp 3 - Ablación +N1 |

In [ ]:
async def clasificar(description: str, bulletin: str, use_n1_context: bool = False) -> ClassifierOutput:
    """Clasifica una publicación individual y devuelve el output estructurado."""
    # Construir el mensaje de usuario con contexto N0 (ámbito geográfico)
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"

    # Opcionalmente añadir la clasificación N1 (tipo de acto por reglas) como pista al LLM
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"

    result = await agent.run(user_msg)
    return result.output


In [ ]:
# Casos cualitativos - 5 ejemplos manuales para validar el agente antes del batch
# Útil para detectar problemas de prompt antes de lanzar los 100 registros
# Casos cualitativos - 5 ejemplos de validación
casos = [
    ("Resolución de 12 de marzo de 2025, de la Dirección General de Calidad y Evaluación "
     "Ambiental, por la que se formula la declaración de impacto ambiental del proyecto "
     "Planta Solar Fotovoltaica Los Llanos, en la provincia de Cáceres.", "doe"),
    ("Resolución de 5 de febrero de 2025, de la Dirección General de Política Energética, "
     "por la que se otorga autorización administrativa previa y de construcción para el "
     "Parque Eólico Sierra Norte, de 48 MW, en Salamanca.", "boe"),
    ("Resolución de 18 de enero de 2025, de la Delegación Territorial de Medio Ambiente, "
     "por la que se otorga autorización ambiental unificada para la planta de biogás "
     "Valdecorneja, en Ávila.", "boja"),
    ("Resolución de 3 de marzo de 2025, de la Universidad de Salamanca, por la que se "
     "convoca concurso-oposición para cubrir plazas de profesor ayudante doctor.", "bocyl"),
    ("Resolución de 21 de febrero de 2025, de la Dirección General de Medio Natural, "
     "por la que se formula el informe de impacto ambiental del proyecto de línea "
     "eléctrica subterránea de 132 kV en Zaragoza.", "boa"),
]

EXPECTED = [
    {"procedures": {"DIA"}, "technologies": {"fotovoltaica"}, "relevant": True},
    {"procedures": {"AAP","AAC"}, "technologies": {"eólica"}, "relevant": True},
    {"procedures": {"AAU"}, "technologies": {"biogás_biometano"}, "relevant": True},
    {"procedures": set(), "technologies": set(), "relevant": False},
    {"procedures": {"IIA"}, "technologies": {"línea_eléctrica"}, "relevant": True},
]

print("Validación cualitativa - 5 casos\n")
aciertos = 0
for i, (desc, bul) in enumerate(casos, 1):
    r = await clasificar(desc, bul)
    pred_proc = set(p.value for p in r.procedures)
    pred_tech = set(t.value for t in r.technologies)
    exp = EXPECTED[i-1]
    ok = (r.is_relevant == exp["relevant"] and pred_proc == exp["procedures"])
    aciertos += ok
    mark = "✅" if ok else "❌"
    print(f"{mark} Caso {i} | is_relevant={r.is_relevant} | procedures={pred_proc} | technologies={pred_tech}")
    if not ok:
        print(f"   Esperado: relevant={exp['relevant']} procedures={exp['procedures']}")
print(f"\nResultado: {aciertos}/5 correctos")

---

##  5. Experimento 1 - Baseline

Configuración zero-shot sin contexto N1. El LLM recibe solo `description` + `bulletin` y debe inferir todos los campos por sí solo.

**Modelo**: Qwen 3.5 9B (thinking desactivado)
**Registros**: 100 (muestra estratificada del ground truth)
**Concurrencia**: 1 (DEC-009: modelos locales procesan secuencialmente)

In [ ]:
from tqdm.asyncio import tqdm_asyncio


# ── clasificar_async ──────────────────────────────────────────────────────────
# Versión async de clasificar() orientada a ejecución en lote.
# Devuelve un dict plano listo para añadir como fila al CSV de resultados.
# Si el LLM falla (timeout, schema inválido tras retries...) captura el error
# y devuelve una fila de error en lugar de interrumpir todo el experimento.
async def clasificar_async(
    description: str,
    bulletin: str,
    use_n1_context: bool = False,
) -> dict:
    # Construir el mensaje de usuario (igual que en clasificar())
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"

    try:
        result = await agent.run(user_msg)
        output = result.output

        # Serializar a dict: los enums se convierten a string, las listas a JSON
        return {
            "is_relevant_pred":  output.is_relevant,
            "act_type_pred":     output.act_type.value,
            "procedures_pred":   json.dumps([p.value for p in output.procedures],   ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence":        output.confidence,
            "reasoning":         output.reasoning,
        }

    except Exception as e:
        # Fila de error - no rompe el bucle, se puede identificar después
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


In [ ]:
#  run_experiment 
# Ejecuta clasificar_async sobre cada fila del DataFrame de entrada.
# - concurrency: cuántas llamadas al LLM van en paralelo (1 para modelos locales)
# - output_path: el CSV se guarda al terminar; si se interrumpe se pierde
async def run_experiment(
    df_input: pd.DataFrame,
    use_n1_context: bool = False,
    concurrency: int = 1,
    output_path: str = "../results/experiment.csv",
) -> pd.DataFrame:
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)

    # Semáforo: limita cuántas corrutinas corren a la vez
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async(row["description"], row["bulletin"], use_n1_context)
            # Combinar columnas originales + predicciones en una sola fila
            return {**row.to_dict(), **pred}

    # Lanzar todas las tareas y esperar con barra de progreso
    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando")

    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)

    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


In [ ]:
# Ejecutar Experimento 1 - Baseline
# Requiere LM Studio activo con el modelo cargado
df_exp1 = await run_experiment(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp1_baseline_qwen9b.csv",
)

print(df_exp1[["grupo_muestreo", "procedures_pred", "confidence"]].head(10))


---

##  6. Análisis de errores - Baseline

Evaluamos las predicciones del Experimento 1 contra las etiquetas manuales del ground truth. El objetivo es identificar patrones de error sistemáticos que guíen la mejora del prompt en  7.

In [ ]:
#  parse_labels 
# Convierte cualquier representación de etiquetas a un set de strings.
# Admite dos formatos:
#   - JSON list:  '["DIA","AAP"]'  →  {'DIA', 'AAP'}
#   - CSV string: 'DIA,AAP'         →  {'DIA', 'AAP'}
# El ground truth usa CSV; las predicciones del agente usan JSON.
def parse_labels(value) -> set:
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    v = str(value).strip()
    # Intentar parsear como JSON primero
    if v.startswith("["):
        try:
            return set(json.loads(v))
        except Exception:
            pass
    # Fallback: split por comas
    return set(x.strip() for x in v.split(",") if x.strip())


In [ ]:
#  compute_metrics 
# Calcula métricas completas de clasificación sobre un DataFrame mergeado:
#   - is_relevant: precisión, recall, F1, matriz de confusión
#   - N2 (procedimientos): P/R/F1 por etiqueta + Macro-F1
#   - Exact match: qué fracción de registros tienen el set de N2 exactamente correcto
#   - Confianza: distribución de la columna 'confidence' reportada por el LLM
def compute_metrics(df_eval):
    y_true = df_eval["is_relevant_gt"].astype(bool)
    y_pred = df_eval["is_relevant_pred"].fillna(False).astype(bool)

    #  is_relevant 
    tp=((y_true)&(y_pred)).sum(); fp=((~y_true)&(y_pred)).sum()
    fn=((y_true)&(~y_pred)).sum(); tn=((~y_true)&(~y_pred)).sum()
    p=tp/(tp+fp) if tp+fp>0 else 0; r=tp/(tp+fn) if tp+fn>0 else 0
    f1_rel=2*p*r/(p+r) if p+r>0 else 0
    print(f"── is_relevant  P={p:.3f}  R={r:.3f}  F1={f1_rel:.3f}  TP={tp} FP={fp} FN={fn} TN={tn}\n")

    #  N2 - métricas por etiqueta 
    N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]
    print(f"{'Label':<8} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5} {'TP':>4} {'FP':>4} {'FN':>4}")
    print("─" * 55)
    macro = 0
    for label in N2:
        yt = df_eval.apply(lambda r: label in parse_labels(r["procedures_gt"]), axis=1)
        yp = df_eval.apply(lambda r: label in parse_labels(r["procedures_pred"]), axis=1)
        tp2=(yt&yp).sum(); fp2=(~yt&yp).sum(); fn2=(yt&~yp).sum()
        p2=tp2/(tp2+fp2) if tp2+fp2>0 else 0
        r2=tp2/(tp2+fn2) if tp2+fn2>0 else 0
        f12=2*p2*r2/(p2+r2) if p2+r2>0 else 0
        sup=yt.sum(); macro+=f12
        print(f"{label:<8} {p2:>6.3f} {r2:>6.3f} {f12:>6.3f} {sup:>5} {tp2:>4} {fp2:>4} {fn2:>4}")
    print("─" * 55)
    print(f"{'Macro-F1':<8} {macro/len(N2):>6.3f}")

    #  Exact match 
    # Consideramos 'exacto' cuando el set de procedimientos predicho == ground truth
    exact = df_eval.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )
    rel = y_true
    print(f"\nExact match (relevantes): {exact[rel].mean():.3f}  ({exact[rel].sum()}/{rel.sum()})")
    print(f"Exact match (todos):      {exact.mean():.3f}  ({exact.sum()}/{len(df_eval)})")

    #  Confianza
    conf = df_eval["confidence"].dropna()
    print(f"\nConfianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return exact, rel


In [ ]:
# Cargar resultados del Experimento 1 y hacer merge con el ground truth
df_exp1 = pd.read_csv("../results/exp1_baseline_qwen9b.csv")

# Merge por descripción: alinea predicciones con etiquetas manuales
df_eval1 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp1[["description","is_relevant_pred","act_type_pred","procedures_pred",
              "technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print(f"Registros evaluados: {len(df_eval1)}")
print(f"Predicciones válidas: {df_eval1['is_relevant_pred'].notna().sum()}")
print(f"Errores: {df_eval1['reasoning'].astype(str).str.startswith('ERROR').sum()}")
print()

print("── Experimento 1 - Baseline zero-shot ──────────────────────────")
exact1, rel1 = compute_metrics(df_eval1)


In [ ]:
# Análisis detallado de errores N2 en registros relevantes
# Solo mostramos los errores en registros que el ground truth marca como relevantes,
# ya que son los que más impactan en la capacidad del sistema de no perder autorizaciones.
print("── Errores N2 en registros relevantes ──────────────────────────────\n")
errores1 = df_eval1[~exact1 & rel1]
print(f"Total errores: {len(errores1)}\n")

for _, row in errores1.iterrows():
    gt = parse_labels(row["procedures_gt"])
    pred = parse_labels(row["procedures_pred"])
    missing = sorted(gt - pred)   # etiquetas que el modelo no detectó
    extra = sorted(pred - gt)     # etiquetas que el modelo añadió por error
    print(f"ID {row['id']} | {str(row.get('bulletin','')[:10])}")
    print(f"  GT  : {sorted(gt)}")
    print(f"  Pred: {sorted(pred)}")
    if missing: print(f"  ❌ Missing: {missing}")
    if extra:   print(f"  ⚠️  Extra:   {extra}")
    print(f"  Razonamiento: {str(row.get('reasoning',''))[:120]}")
    print()


---

##  7. Prompt v2 - Mejora basada en errores

Los cambios respecto al Prompt v1 se agrupan en tres categorías:

**Cambios consolidados** - basados en errores sistemáticos del Experimento 1:
- Regla explícita AAI+DIA: cuando una resolución formula DIA y otorga AAI en el mismo acto → [DIA, AAI]
- Refuerzo de denegaciones/desistimientos con ejemplo concreto de DUP

**Cambios experimentales** - a validar con el tutor/equipo (PENDIENTE-002):
- Regla: "todo procedimiento N2 identificado implica is_relevant=True, independientemente del tipo de proyecto" → esto incluye IIA de sondeos de agua, IAE de planes urbanísticos, AAI de cementeras. El Baseline los marcaba como False coherentemente con una interpretación estricta del dominio energético. Aquí asumimos scope amplio para medir el impacto - si el cliente prefiere scope estricto, esta regla se elimina y las métricas del Baseline en IIA/IAE/AAI mejorarían.
- DIA de proyectos no energéticos (concentración parcelaria, agroturismo) → is_relevant=True

**Sin cambios** - reglas del v1 que funcionaron bien:
- AAP, AAC, AAU, DUP → F1 perfecto o casi perfecto, no tocar
- Combinaciones AAP+AAC y AAP+AAC+DUP → ya funcionan

In [ ]:
# System prompt v2 - version mejorada del prompt baseline
# Cambios principales: definición más precisa de relevancia, ejemplos de
# casos frontera (sondeos de agua, planes urbanísticos) y énfasis en que
# el tipo de proyecto no determina la relevancia, solo el procedimiento N2.
SYSTEM_PROMPT_V2 = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles.
Tu tarea es analizar la descripción de una publicación y asignarle etiquetas según la taxonomía definida.

## Dominio
Las publicaciones relevantes son aquellas que contienen al menos un procedimiento N2.
El tipo de proyecto NO determina la relevancia - una IIA sobre un sondeo de agua,
una IAE sobre un plan urbanístico o una AAI sobre una cementera son igualmente relevantes.
Son is_relevant=False: RRHH, contratos, subvenciones, licitaciones, padrones fiscales,
convenios de transporte, telecomunicaciones, plantillas orgánicas.

## Procedimientos N2
| Etiqueta | Descripción |
|----------|-------------|
| DIA | Declaración de Impacto Ambiental - resolución que formula o aprueba el impacto ambiental |
| AAP | Autorización Administrativa Previa - valida el anteproyecto |
| AAC | Autorización Administrativa de Construcción - permiso definitivo de obras |
| AAU | Autorización Ambiental Unificada - equivalente regional a DIA en BOJA/DOE/BON |
| IIA | Informe de Impacto Ambiental - evaluación simplificada, distinta de DIA |
| AAI | Autorización Ambiental Integrada - permiso IPPC/IED, distinta de DIA |
| IAE | Informe/Declaración Ambiental Estratégico - aplica a planes y programas |
| DUP | Declaración de Utilidad Pública - reconoce interés general, habilita expropiación |

## Tecnologías N3
fotovoltaica · eólica · almacenamiento · hibridación · hidroeléctrica ·
biogás_biometano · biomasa · hidrógeno · línea_eléctrica · gas_natural · petróleo
Si el proyecto no es energético, technologies=[]

## Reglas críticas
1. is_relevant=True si y solo si identificas al menos un procedimiento N2 - independientemente del tipo de proyecto
2. AAU ≠ DIA - son procedimientos distintos aunque equivalentes funcionalmente
3. "autorización administrativa previa y de construcción" → [AAP, AAC] (no solo AAP)
4. "aprobación del proyecto de ejecución" junto a AAP → añadir AAC
5. IIA ≠ DIA - el informe de impacto ambiental es evaluación simplificada
6. AAI ≠ DIA - solo añadir DIA si el texto menciona EXPLÍCITAMENTE "declaración de impacto ambiental"
7. Cuando una resolución formula DIA Y otorga AAI en el mismo acto → [DIA, AAI]
8. IAE aplica a planes y programas, no a proyectos individuales
9. DUP puede acompañar a AAP/AAC pero no es AAP ni AAC por sí sola
10. Denegaciones y desistimientos heredan el tipo del procedimiento - ejemplo: "se da por desistido el titular de AAP+AAC+DUP" → [AAP, AAC, DUP]
11. Las modificaciones heredan los procedimientos del acto modificado
12. reasoning debe citar el fragmento exacto del texto que dispara cada etiqueta
""".strip()

---

##  8. Experimento 2 - Prompt v2

Mismo modelo (Qwen 3.5 9B), mismos 100 registros, pero con el prompt mejorado. Comparamos con Experimento 1 para cuantificar la ganancia del prompt engineering.

In [ ]:
# Agente v2 - mismo modelo, prompt mejorado
agent_v2 = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT_V2,
)


In [ ]:
# ── clasificar_async_v2 ──────────────────────────────────────────────────────
# Idéntica a clasificar_async pero usa agent_v2 (prompt v2).
# Se define por separado para poder comparar ambos experimentos en el mismo kernel.
async def clasificar_async_v2(description: str, bulletin: str, use_n1_context: bool = False) -> dict:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    try:
        result = await agent_v2.run(user_msg)
        output = result.output
        return {
            "is_relevant_pred":  output.is_relevant,
            "act_type_pred":     output.act_type.value,
            "procedures_pred":   json.dumps([p.value for p in output.procedures],   ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence":        output.confidence,
            "reasoning":         output.reasoning,
        }
    except Exception as e:
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


In [ ]:
#  run_experiment_v2 
# Mismo mecanismo que run_experiment pero llama a clasificar_async_v2.
async def run_experiment_v2(df_input, use_n1_context=False, concurrency=1, output_path="../results/exp.csv"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async_v2(row["description"], row["bulletin"], use_n1_context)
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando v2")
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)
    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


In [ ]:
# Ejecutar Experimento 2 - Prompt v2, sin contexto N1
df_exp2 = await run_experiment_v2(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp2_promptv2_qwen9b.csv",
)


In [ ]:
# Evaluar Experimento 2 - Prompt v2
df_exp2 = pd.read_csv("../results/exp2_promptv2_qwen9b.csv")
df_eval2 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp2[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 2 - Prompt v2 ──────────────────────────")
exact2, rel2 = compute_metrics(df_eval2)


---

##  9. Experimento 3 - Ablación +N1

Añadimos el `act_type` pre-computado por reglas como contexto al prompt. Pregunta de investigación: **¿cuánto aporta saber la forma jurídica del documento para clasificar el procedimiento N2?**

Usamos Prompt v2 + contexto N1.

In [ ]:
# Ejecutar Experimento 3 - Prompt v2 + contexto N1
# use_n1_context=True añade el tipo de acto inferido por reglas al mensaje del LLM
df_exp3 = await run_experiment_v2(
    df_anotado,
    use_n1_context=True,  # ← única diferencia respecto al Experimento 2
    concurrency=1,
    output_path="../results/exp3_promptv2_n1_qwen9b.csv",
)


In [ ]:
# Evaluar Experimento 3 - Prompt v2 + contexto N1
df_exp3 = pd.read_csv("../results/exp3_promptv2_n1_qwen9b.csv")
df_eval3 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp3[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 3 - Prompt v2 + N1 ────────────────────")
exact3, rel3 = compute_metrics(df_eval3)


---

##  10. Experimento 4 - Few-shot

Añadimos ejemplos reales al prompt (one o two-shot por categoría difícil). Pregunta de investigación: **¿mejora la clasificación de los casos borde cuando el modelo tiene ejemplos concretos?**

Los ejemplos se seleccionan de los errores identificados en  6.

> ⚙️ **Pendiente**: implementar tras analizar los errores del Experimento 1.

In [ ]:
# System prompt v3 - Few-shot
# Añade ejemplos concretos de publicaciones relevantes e irrelevantes para
# guiar al LLM mediante in-context learning. Los ejemplos cubren los casos
# más frecuentes de confusión detectados en los Experimentos 1 y 2.
SYSTEM_PROMPT_V3 = """
Eres un experto en clasificación de publicaciones de boletines oficiales españoles.
Tu tarea es analizar la descripción de una publicación y asignarle etiquetas según la taxonomía definida.

## Dominio
Las publicaciones relevantes son aquellas que contienen al menos un procedimiento N2.
El tipo de proyecto NO determina la relevancia - una IIA sobre un sondeo de agua,
una IAE sobre un plan urbanístico o una AAI sobre una cementera son igualmente relevantes.
Son is_relevant=False: RRHH, contratos, subvenciones, licitaciones, padrones fiscales,
convenios de transporte, telecomunicaciones, plantillas orgánicas.

## Procedimientos N2
| Etiqueta | Descripción |
|----------|-------------|
| DIA | Declaración de Impacto Ambiental - resolución que formula o aprueba el impacto ambiental |
| AAP | Autorización Administrativa Previa - valida el anteproyecto |
| AAC | Autorización Administrativa de Construcción - permiso definitivo de obras |
| AAU | Autorización Ambiental Unificada - equivalente regional a DIA en BOJA/DOE/BON |
| IIA | Informe de Impacto Ambiental - evaluación simplificada, distinta de DIA |
| AAI | Autorización Ambiental Integrada - permiso IPPC/IED, distinta de DIA |
| IAE | Informe/Declaración Ambiental Estratégico - aplica a planes y programas |
| DUP | Declaración de Utilidad Pública - reconoce interés general, habilita expropiación |

## Tecnologías N3
fotovoltaica · eólica · almacenamiento · hibridación · hidroeléctrica ·
biogás_biometano · biomasa · hidrógeno · línea_eléctrica · gas_natural · petróleo
Si el proyecto no es energético, technologies=[]

## Reglas críticas
1. is_relevant=True si y solo si identificas al menos uno de estos procedimientos:
   DIA, AAP, AAC, AAU, IIA, AAI, IAE o DUP - independientemente del tipo de proyecto
2. AAU ≠ DIA - son procedimientos distintos aunque equivalentes funcionalmente
3. "autorización administrativa previa y de construcción" → [AAP, AAC] (no solo AAP)
4. "aprobación del proyecto de ejecución" junto a AAP → añadir AAC
5. IIA ≠ DIA - el informe de impacto ambiental es evaluación simplificada
6. AAI ≠ DIA - solo añadir DIA si el texto menciona EXPLÍCITAMENTE "declaración de impacto ambiental"
7. Cuando una resolución formula DIA Y otorga o modifica AAI en el mismo acto → [DIA, AAI]
8. IAE aplica a planes y programas, no a proyectos individuales
9. DUP puede acompañar a AAP/AAC pero no es AAP ni AAC por sí sola
10. Denegaciones y desistimientos heredan el tipo del procedimiento - ejemplo: "se da por desistido el titular de AAP+AAC+DUP" → [AAP, AAC, DUP]
11. Las modificaciones heredan los procedimientos del acto modificado
12. Los ANUNCIOS de información pública sobre solicitudes son tan relevantes como las resoluciones - etiquetar según los procedimientos que mencionen
13. reasoning debe citar el fragmento exacto del texto que dispara cada etiqueta

## Ejemplos

### Ejemplo 1 - AAI simplificada (is_relevant=True aunque el proyecto no sea energético)
Descripción: «Anuncio por el que se hace pública la Resolución de la Consejería de Transición
Ecológica, Industria y Comercio, de otorgamiento de la autorización ambiental integrada
simplificada a la instalación de "fabricación de hormigones frescos" del titular Cementos
Secil, S.L.U., ubicada en polígono industrial La Curiscada (Tineo).»
→ is_relevant=True | procedures=[AAI] | technologies=[]
Razón: "autorización ambiental integrada simplificada" es una variante de AAI → relevante aunque sea industria del cemento.

### Ejemplo 2 - DIA + AAI en el mismo acto
Descripción: «Resolución de la Directora General de Armonización Urbanística y Evaluación
ambiental por la que se formula la declaración de impacto ambiental de la modificación
sustancial de la AAI IPPC 02/2015 centro de recepción y pretratamiento de hidrocarburos,
aceites usados y aguas aceitosas en el dique del Oeste a Palma.»
→ is_relevant=True | procedures=[DIA, AAI] | technologies=[petróleo]
Razón: "formula la declaración de impacto ambiental" → DIA. "modificación sustancial de la AAI" → AAI. Ambos en el mismo acto.

### Ejemplo 3 - Anuncio de información pública con AAP+AAC+DUP
Descripción: «Anuncio de 03/01/2025, de la Delegación Provincial de Desarrollo Sostenible
de Cuenca, sobre información pública de la solicitud de autorización administrativa previa,
aprobación del proyecto de ejecución y reconocimiento en concreto de utilidad pública
de la instalación eléctrica de alta tensión.»
→ is_relevant=True | procedures=[AAP, AAC, DUP] | technologies=[línea_eléctrica]
Razón: Los anuncios de solicitud son relevantes. "autorización administrativa previa" → AAP. "aprobación del proyecto de ejecución" → AAC. "reconocimiento en concreto de utilidad pública" → DUP.
""".strip()

In [ ]:
# Agente v3 - mismo modelo, prompt few-shot
agent_v3 = Agent(
    model,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT_V3,
)


In [ ]:
# ── clasificar_async_v3 ──────────────────────────────────────────────────────
# Idéntica a las versiones anteriores pero usa agent_v3 (prompt few-shot).
async def clasificar_async_v3(description: str, bulletin: str, use_n1_context: bool = False) -> dict:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    try:
        result = await agent_v3.run(user_msg)
        output = result.output
        return {
            "is_relevant_pred": output.is_relevant,
            "act_type_pred": output.act_type.value,
            "procedures_pred": json.dumps([p.value for p in output.procedures], ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence": output.confidence,
            "reasoning": output.reasoning,
        }
    except Exception as e:
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


In [ ]:
# ── run_experiment_v3 ────────────────────────────────────────────────────────
async def run_experiment_v3(df_input, use_n1_context=False, concurrency=1, output_path="../results/exp.csv"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async_v3(row["description"], row["bulletin"], use_n1_context)
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando v3")
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)
    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


In [ ]:
# Ejecutar Experimento 4 - Few-shot (Prompt v3)
df_exp4 = await run_experiment_v3(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp4_fewshot_qwen9b.csv",
)


In [ ]:
# Evaluar Experimento 4 - Few-shot (Prompt v3)
df_exp4 = pd.read_csv("../results/exp4_fewshot_qwen9b.csv")
df_eval4 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp4[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 4 - Few-shot (Prompt v3) ──────────────────")
exact4, rel4 = compute_metrics(df_eval4)


---

##  11. Comparativa de modelos

Repetimos el mejor experimento (Prompt v2 + configuración óptima) con Gemma 4 4B.
Pregunta de investigación: **¿cuánto se pierde en F1 al usar un modelo 4B vs 9B?**

Para cambiar al Gemma 4B: en  0, cambiar `LM_STUDIO_MODEL = "gemma-4-4b-it"` y reiniciar el kernel.

> ⚙️ **Pendiente**: ejecutar cuando Gemma 4B esté descargado en LM Studio.

In [ ]:
# ── Experimento 5 - Gemma 4 4B (Prompt v2) ───────────────────────────────────
# Mismo prompt v2, mismo ground truth - solo cambia el modelo.
# Permite comparar Qwen 9B vs Gemma 4B con todas las demás variables fijas.

from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

# Cargar Gemma 4B desde LM Studio (debe estar activo con este modelo)
model_gemma = OpenAIChatModel(
    "gemma-4-e4b-it",
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

agent_gemma = Agent(
    model_gemma,
    output_type=ClassifierOutput,
    system_prompt=SYSTEM_PROMPT_V2,
)


In [ ]:
# ── clasificar_async_gemma ───────────────────────────────────────────────────
# Idéntica a las versiones anteriores pero usa agent_gemma.
async def clasificar_async_gemma(description: str, bulletin: str, use_n1_context: bool = False) -> dict:
    n0 = get_ambito(bulletin)
    user_msg = f"Boletín: {bulletin.upper()} (ámbito: {n0})\n\nDescripción: {description}"
    if use_n1_context:
        act_type_pre = inferir_act_type(description, bulletin)
        user_msg += f"\n\nTipo de acto pre-clasificado (N1): {act_type_pre.value}"
    try:
        result = await agent_gemma.run(user_msg)
        output = result.output
        return {
            "is_relevant_pred":  output.is_relevant,
            "act_type_pred":     output.act_type.value,
            "procedures_pred":   json.dumps([p.value for p in output.procedures],   ensure_ascii=False),
            "technologies_pred": json.dumps([t.value for t in output.technologies], ensure_ascii=False),
            "confidence":        output.confidence,
            "reasoning":         output.reasoning,
        }
    except Exception as e:
        return {
            "is_relevant_pred": None, "act_type_pred": None,
            "procedures_pred": "[]", "technologies_pred": "[]",
            "confidence": None, "reasoning": f"ERROR: {str(e)[:100]}",
        }


In [ ]:
# ── run_experiment_gemma ─────────────────────────────────────────────────────
async def run_experiment_gemma(df_input, use_n1_context=False, concurrency=1, output_path="../results/exp.csv"):
    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    semaphore = asyncio.Semaphore(concurrency)

    async def process_row(row):
        async with semaphore:
            pred = await clasificar_async_gemma(row["description"], row["bulletin"], use_n1_context)
            return {**row.to_dict(), **pred}

    tasks = [process_row(row) for _, row in df_input.iterrows()]
    results = await tqdm_asyncio.gather(*tasks, desc="Clasificando Gemma 4B")
    df_results = pd.DataFrame(results)
    df_results.to_csv(output_path, index=False)
    errores = df_results["reasoning"].astype(str).str.startswith("ERROR").sum()
    print(f"\n✓ Guardado en {output_path} | Errores: {errores}/{len(df_results)}")
    return df_results


In [ ]:
# Ejecutar Experimento 5 - Gemma 4 4B con Prompt v2
df_exp5 = await run_experiment_gemma(
    df_anotado,
    use_n1_context=False,
    concurrency=1,
    output_path="../results/exp5_promptv2_gemma4b.csv",
)


In [ ]:
# Evaluar Experimento 5 - Gemma 4 4B
df_exp5 = pd.read_csv("../results/exp5_promptv2_gemma4b.csv")
df_eval5 = df_anotado[["id","is_relevant_gt","procedures_gt","technologies_gt","description"]].merge(
    df_exp5[["description","is_relevant_pred","procedures_pred","technologies_pred","confidence","reasoning"]],
    on="description", how="left"
)

print("── Experimento 5 - Gemma 4 4B · Prompt v2 ────────────────")
exact5, rel5 = compute_metrics(df_eval5)


---

##  12. Tabla resumen - Comparativa de experimentos

In [ ]:
#  Tabla resumen de todos los experimentos 
# Compara los 5 experimentos en las métricas clave:
#   is_rel F1, Macro-F1 (N2), Exact match y confianza media/mínima
# parse_labels y df_anotado ya están definidos en celdas anteriores

import os

N2 = ["DIA","AAP","AAC","AAU","IIA","AAI","IAE","DUP"]

# Lista de experimentos: (nombre, modelo, configuración, ruta CSV)
experimentos = [
    ("Exp 1 · Baseline zero-shot",   "Qwen 3.5 9B", "Zero-shot",   "../results/exp1_baseline_qwen9b.csv"),
    ("Exp 2 · Prompt v2",            "Qwen 3.5 9B", "Zero-shot",   "../results/exp2_promptv2_qwen9b.csv"),
    ("Exp 3 · v2 + N1",              "Qwen 3.5 9B", "+N1 context", "../results/exp3_promptv2_n1_qwen9b.csv"),
    ("Exp 4 · Few-shot (v3)",         "Qwen 3.5 9B", "Few-shot",    "../results/exp4_fewshot_qwen9b.csv"),
    ("Exp 5 · Gemma 4B · Prompt v2", "Gemma 4 4B",  "Zero-shot",   "../results/exp5_promptv2_gemma4b.csv"),
]

rows = []
for nombre, modelo, config, path in experimentos:
    if not os.path.exists(path):
        print(f"⚠️  {nombre}: archivo no encontrado")
        continue

    # Cargar predicciones y hacer merge con ground truth
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","procedures_gt","description"]].merge(
        df_r[["description","is_relevant_pred","procedures_pred","confidence","reasoning"]],
        on="description", how="left"
    )

    # Métricas is_relevant
    y_true = df_e["is_relevant_gt"].astype(bool)
    y_pred = df_e["is_relevant_pred"].fillna(False).astype(bool)
    tp=((y_true)&(y_pred)).sum(); fp=((~y_true)&(y_pred)).sum()
    fn=((y_true)&(~y_pred)).sum()
    p=tp/(tp+fp) if tp+fp>0 else 0
    r=tp/(tp+fn) if tp+fn>0 else 0
    f1_rel=2*p*r/(p+r) if p+r>0 else 0

    # Macro-F1 sobre las 8 etiquetas N2
    macro = 0
    for label in N2:
        yt = df_e.apply(lambda r: label in parse_labels(r["procedures_gt"]), axis=1)
        yp = df_e.apply(lambda r: label in parse_labels(r["procedures_pred"]), axis=1)
        tp2=(yt&yp).sum(); fp2=(~yt&yp).sum(); fn2=(yt&~yp).sum()
        p2=tp2/(tp2+fp2) if tp2+fp2>0 else 0
        r2=tp2/(tp2+fn2) if tp2+fn2>0 else 0
        macro += 2*p2*r2/(p2+r2) if p2+r2>0 else 0

    # Exact match: fracción con set de N2 exactamente correcto
    exact = df_e.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )
    errores = df_e["reasoning"].astype(str).str.startswith("ERROR").sum()
    conf_mean = df_e["confidence"].mean()
    conf_min = df_e["confidence"].min()

    rows.append({
        "Experimento": nombre,
        "Modelo": modelo,
        "Config": config,
        "is_rel F1": f"{f1_rel:.3f}",
        "Macro-F1": f"{macro/len(N2):.3f}",
        "Exact(rel)": f"{exact[y_true].mean():.3f}",
        "Exact(all)": f"{exact.mean():.3f}",
        "Conf media": f"{conf_mean:.3f}",
        "Conf min": f"{conf_min:.3f}",
        "Errores fmt": errores,
    })

df_summary = pd.DataFrame(rows)
print(df_summary.to_string(index=False))
df_summary.to_csv("../results/tabla_resumen_experimentos.csv", index=False)
print("\n✓ Guardado en ../results/tabla_resumen_experimentos.csv")


---

##  13. Calibración de confianza

Analizamos si el campo `confidence` es un predictor real de calidad. **Hipótesis**: los registros con `confidence < 0.8` deberían tener peor F1 que los de `confidence > 0.95`.

Si el modelo es sobreconfiante (todos los valores entre 0.95-1.0), el campo `confidence` no sirve como filtro práctico - esto es un hallazgo relevante para la memoria.

In [ ]:
# ── Calibración comparativa entre experimentos ───────────────────────────────
# Mide si la confianza reportada por el LLM es un predictor real de la exactitud.
# Un modelo bien calibrado tiene mayor accuracy cuando reporta confianza alta.
# 'Predictor' = diferencia de >0.1 en exact match entre confianza ≥0.95 y <0.95.
print(" Calibración comparativa ")
print()

calibracion = [
    ("Qwen 3.5 9B · Baseline",  "../results/exp1_baseline_qwen9b.csv"),
    ("Qwen 3.5 9B · Prompt v2", "../results/exp2_promptv2_qwen9b.csv"),
    ("Qwen 3.5 9B · +N1",       "../results/exp3_promptv2_n1_qwen9b.csv"),
    ("Qwen 3.5 9B · Few-shot",  "../results/exp4_fewshot_qwen9b.csv"),
    ("Gemma 4 4B · Prompt v2",  "../results/exp5_promptv2_gemma4b.csv"),
]

print(f"{'Experimento':<30} {'Media':>7} {'Min':>7} {'>=0.95':>8} {'Predictor?':>12}")
print("─" * 70)

for nombre, path in calibracion:
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","procedures_gt","description"]].merge(
        df_r[["description","is_relevant_pred","procedures_pred","confidence"]],
        on="description", how="left"
    )
    # Solo filas con confianza válida
    df_e = df_e[df_e["confidence"].notna()]
    df_e["exact"] = df_e.apply(
        lambda r: parse_labels(r["procedures_gt"]) == parse_labels(r["procedures_pred"]), axis=1
    )

    conf = df_e["confidence"]
    alta = df_e[df_e["confidence"] >= 0.95]["exact"].mean()
    baja_n = (df_e["confidence"] < 0.95).sum()
    baja_acc = df_e[df_e["confidence"] < 0.95]["exact"].mean() if baja_n > 0 else float("nan")
    sobreconf = (conf >= 0.95).mean()

    # Determinar si la confianza es un predictor estadísticamente útil
    if baja_n >= 3 and not pd.isna(baja_acc):
        predictor = "✅ Sí" if (alta - baja_acc) > 0.1 else "❌ No"
    else:
        predictor = "- insuf."

    print(f"{nombre:<30} {conf.mean():>7.3f} {conf.min():>7.3f} {sobreconf:>7.1%}  {predictor:>12}")

print()
print("Conclusión: solo Gemma 4B tiene distribución de confianza suficientemente")
print("dispersa para usarse como señal de revisión manual.")


---

##  14. Conclusiones y trabajo futuro

### Hallazgos principales

> *Completar tras ejecutar todos los experimentos*

### Limitaciones

1. **Ground truth Opción A**: el dataset de 100 registros fue anotado manualmente sin revisión cruzada. La Opción B (anotación rigurosa con múltiples anotadores) queda como trabajo futuro.
2. **Modelo local limitado**: Qwen 3.5 9B con 16GB RAM es el modelo más grande ejecutable en el hardware disponible. Los experimentos con modelos frontera (Gemini 2.5 Pro) se limitan a mini-muestras por rate limits.
3. **Sobreconfianza**: el campo `confidence` no es un predictor fiable de calidad en modelos locales - todos los valores tienden a 0.95-1.0.

### Trabajo futuro - v2

| Tarea | Descripción |
|-------|-------------|
| Ampliar corpus | Extender a las 18 familias y 92 procedimientos del N2 completo |
| Ground truth riguroso | Opción B: anotación manual con revisión cruzada |
| Nuevas tecnologías N3 | `industria_ippc`, `infraestructura_hidrica`, `ordenacion_territorial`, `turismo_edificacion` |
| N1 para RRHH | LLM zero-shot para registros sin tipo de acto explícito |
| Modelos frontera | Evaluación completa con Gemini 2.5 Pro / GPT-4o |